[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_73_Alerting_SLOs_OnCall.ipynb)

# Lesson 73 — Alerting, SLOs & On-Call
### Phase 8: Production Ops for LLM Systems · Lesson 3 of ~6

You now have two of the three pillars of production observability:

| Lesson | Pillar | What it answers |
|---|---|---|
| **L71** | Metrics | *Is something wrong?* (error rate, p95, cost) |
| **L72** | Trace search | *What is wrong, and for whom?* (read one request end-to-end) |
| **L73 (today)** | **Alerting + SLOs** | ***When do we wake a human, and what did we even promise?*** |

The slogan to keep in your head:

> **Metrics measure. Trace search diagnoses. But an SLO defines the promise, and an alert decides when a human gets pulled out of dinner.**

A dashboard nobody is staring at 24/7 catches nothing. The job of this lesson is to turn your L71 metrics into a small number of *trustworthy* signals — signals that fire on real, actionable problems and stay quiet otherwise. That last part is the hard part: an alerting system that cries wolf is worse than none, because your on-call engineer learns to ignore it.

**Phase 8 roadmap (tentative, adapts to your questions):**

| # | Topic | Status |
|---|---|---|
| L71 | Observability & online evals | done |
| L72 | Structured logging & trace search | done |
| **L73** | **Alerting, SLOs & on-call** | **today** |
| L74 | A/B testing & guarded rollouts | next |
| L75 | Feedback loops & the data flywheel | planned |
| L76 | Capstone: ship a prod-obs module for `agent-bench` | planned |

Everything runs **offline and deterministically** — no API key needed. We operate purely on events you already log.

## 1. The vocabulary you must not confuse: SLI, SLO, SLA, error budget

These four words get thrown around interchangeably and it causes real confusion. Pin them down once:

| Term | What it is | Example | Who cares |
|---|---|---|---|
| **SLI** — *Indicator* | A **measured number**. A ratio of good events to total. | "99.2% of agent runs succeeded today" | you, right now |
| **SLO** — *Objective* | A **target** you promise yourself the SLI will meet. | "≥ 99% success over 30 days" | your team |
| **SLA** — *Agreement* | A **contract** with a customer, with penalties if you miss. | "99% or we refund 10%" | legal / sales |
| **Error budget** | `100% − SLO`. The failure you are *allowed*. | 1% of all requests may fail | everyone |

The single most important mental shift in this whole lesson:

> **Reliability is not "zero errors." It is a *budget you are allowed to spend.***

If your SLO is 99% success over 30 days and you serve 1,000,000 requests, your **error budget is 10,000 failed requests**. As long as you have budget left, a failure is *normal and expected* — not an emergency. This reframing is what stops teams from paging themselves at 3am over a single blip. You only escalate when you are **burning the budget too fast to survive the window**.

That rate — how fast you are spending the budget — is called the **burn rate**, and it is the heart of good alerting. We build up to it step by step.

In [ ]:
# --- Setup: deterministic, offline, no API key ------------------------------
# We install only 'rich' for pretty tables. Everything else is stdlib.
!pip install rich -q

import random, math
from dataclasses import dataclass, field
from enum import Enum
from datetime import datetime, timedelta, timezone
from typing import Optional
from rich.console import Console
from rich.table import Table

# force_jupyter=False keeps Rich output clean & copyable in Colab and when the
# notebook is executed headless during validation (a habit from L64).
console = Console(force_jupyter=False, no_color=False, highlight=False)

# BASE is where we write our reusable module later. In Colab this is /content.
# (During offline validation this single line is swapped to a temp dir.)
BASE = '/content'

# A FIXED "now" makes every time-window query 100% reproducible run to run.
UTC = timezone.utc
NOW = datetime(2026, 7, 19, 12, 0, 0, tzinfo=UTC)
SEED = 73
random.seed(SEED)

print('Setup OK. NOW =', NOW.isoformat(), '| BASE =', BASE)

## 2. From raw events to an SLI

We simulate the last **3 hours** of production traffic for an agent endpoint: ~30 requests per minute, each either OK or an error, with a latency. Baseline is healthy (~1% errors). We **bake in one incident**: for a 30-minute block a downstream dependency degrades and ~40% of requests start failing.

This mirrors the deterministic-traffic generators from L71/L72 — the point is to have a *findable* problem so we can later assert that our alerting engine catches it **and** stays quiet outside it.

In [ ]:
# --- Generate 3 hours of deterministic traffic with one baked-in incident ---
DURATION_MIN   = 180                 # 3 hours, minute 0 = oldest, 179 = newest
REQ_PER_MIN    = 30
BASELINE_ERR   = 0.01                # 1% healthy failure rate
INCIDENT_START = 100                 # minute the dependency starts degrading
INCIDENT_END   = 130                 # minute it recovers (exclusive)
INCIDENT_ERR   = 0.40                # 40% failures during the incident

@dataclass
class Event:
    minute: int          # minutes since window start (0..179)
    ts: datetime         # absolute timestamp
    ok: bool
    latency_ms: float

def simulate():
    events = []
    window_start = NOW - timedelta(minutes=DURATION_MIN)
    for minute in range(DURATION_MIN):
        in_incident = INCIDENT_START <= minute < INCIDENT_END
        err_rate = INCIDENT_ERR if in_incident else BASELINE_ERR
        for _ in range(REQ_PER_MIN):
            ok = random.random() >= err_rate
            # latency: healthy ~450ms, incident inflates the tail
            base = random.gauss(450, 80)
            latency = base + (random.gauss(900, 300) if in_incident else 0)
            ts = window_start + timedelta(minutes=minute,
                                          seconds=random.uniform(0, 59))
            events.append(Event(minute, ts, ok, max(1.0, latency)))
    return events

EVENTS = simulate()
total = len(EVENTS)
bad = sum(1 for e in EVENTS if not e.ok)
print(f'Generated {total} events over {DURATION_MIN} min '
      f'({REQ_PER_MIN}/min).')
print(f'Total failures: {bad}  (overall error rate {bad/total:.2%})')
print(f'Incident baked in at minutes {INCIDENT_START}-{INCIDENT_END} '
      f'(~{INCIDENT_ERR:.0%} errors).')

## 3. SLO, SLI and the error budget

Now we attach a **promise** to this traffic: 99% of requests should succeed. From that promise the **error budget** falls out automatically, and we can ask the two questions that actually matter to a team:

1. **Are we meeting the SLO** over the compliance window?
2. **How much budget is left?** (If it is negative, we already blew it.)

In [ ]:
# --- SLO + error budget ------------------------------------------------------
@dataclass
class SLO:
    name: str
    target: float          # 0.99 = promise 99% success
    window_minutes: int

    @property
    def allowed_error_rate(self):
        return 1.0 - self.target        # 99% -> 1% may fail

def sli_success_rate(n_total, n_bad):
    if n_total <= 0:
        return 1.0
    return 1.0 - (n_bad / n_total)

def error_budget(slo, n_total, n_bad):
    budget = slo.allowed_error_rate * n_total      # failures we are allowed
    spent  = float(n_bad)                          # failures we actually had
    remaining = budget - spent
    return {
        'budget_events': budget,
        'spent_events': spent,
        'remaining_events': remaining,
        'remaining_frac': (remaining / budget) if budget > 0 else 0.0,
        'sli': sli_success_rate(n_total, n_bad),
        'slo_met': sli_success_rate(n_total, n_bad) >= slo.target,
    }

SLO_MAIN = SLO(name='agent_success', target=0.99, window_minutes=DURATION_MIN)
eb = error_budget(SLO_MAIN, total, bad)

t = Table(title='SLO status over the full 3h window', show_lines=False)
t.add_column('metric'); t.add_column('value', justify='right')
t.add_row('SLO target', f'{SLO_MAIN.target:.1%} success')
t.add_row('measured SLI', f'{eb["sli"]:.2%} success')
t.add_row('SLO met?', 'YES' if eb['slo_met'] else 'NO')
t.add_row('error budget (events)', f'{eb["budget_events"]:.0f}')
t.add_row('budget spent (events)', f'{eb["spent_events"]:.0f}')
t.add_row('budget remaining', f'{eb["remaining_events"]:.0f} '
          f'({eb["remaining_frac"]:.0%})')
console.print(t)

# Self-test: with a 40%-error incident on top of 1% baseline, a 99% SLO
# is blown for this window and the budget goes negative.
assert eb['slo_met'] is False, 'incident should blow the 99% SLO'
assert eb['remaining_events'] < 0, 'budget should be overspent'
print('\nSelf-test passed: incident overspends the error budget.')

## 4. Why a naive threshold alert is a trap — enter burn rate

The obvious first instinct: **"page me when the error rate goes above 5%."** It seems reasonable. It is also how teams end up ignoring their pager. Three problems:

- **It flaps.** A metric hovering near 5% crosses the line up and down every minute, firing a storm of on/off alerts.
- **It has no notion of the promise.** 5% might be catastrophic for a 99.9% SLO and totally fine for a 95% one.
- **It ignores *duration*.** A 2-second blip at 6% is noise; 6% sustained for an hour is an outage. A raw threshold can't tell them apart.

**Burn rate** fixes the second problem. It measures how fast you are spending the error budget, *relative to the promise*:

```
burn_rate = observed_error_rate / allowed_error_rate
```

A burn rate of **1.0** means you'd spend exactly your whole budget over the compliance window — sustainable. A burn rate of **14.4** over one hour means you'd burn a **30-day** budget in about **2 days** — that's Google SRE's classic "page now" threshold. The beauty: the same burn-rate thresholds work for *any* SLO, because the promise is baked into the denominator.

In [ ]:
# --- Burn rate, computed on a rolling window ---------------------------------
def burn_rate(slo, window_error_rate):
    allowed = slo.allowed_error_rate
    if allowed <= 0:
        return float('inf') if window_error_rate > 0 else 0.0
    return window_error_rate / allowed

def error_rate_in_window(events, end_minute, width_min):
    # error rate over [end_minute-width_min, end_minute)
    lo = end_minute - width_min
    win = [e for e in events if lo <= e.minute < end_minute]
    if not win:
        return 0.0, 0
    bad_w = sum(1 for e in win if not e.ok)
    return bad_w / len(win), len(win)

# Sample the burn rate at a healthy minute vs mid-incident, on a 5-min window.
for label, m in [('healthy (min 60)', 60), ('mid-incident (min 120)', 120)]:
    er, n = error_rate_in_window(EVENTS, m, 5)
    br = burn_rate(SLO_MAIN, er)
    print(f'{label:24s}  err={er:6.2%}  burn_rate={br:6.1f}x  (n={n})')

# Self-test: burn rate must be dramatically higher inside the incident.
er_heal, _ = error_rate_in_window(EVENTS, 60, 5)
er_inc, _  = error_rate_in_window(EVENTS, 120, 5)
assert burn_rate(SLO_MAIN, er_inc) > 10 * max(burn_rate(SLO_MAIN, er_heal), 0.1)
print('\nSelf-test passed: incident burn rate dwarfs the healthy burn rate.')

## 5. Killing the flapping: for-duration and hysteresis

Burn rate tells us *how bad*. It still doesn't stop **flapping** — a value dancing across the threshold. Two standard tricks, both of which you'll see in Prometheus/Alertmanager and every mature alerting stack:

**1. `for` duration (a.k.a. "pending").** Don't fire the instant the condition is true — require it to hold for N consecutive evaluations first. This turns single-sample spikes into non-events. The alert sits in a **PENDING** state while it waits, and only escalates to **FIRING** if the condition persists.

**2. Hysteresis (two thresholds).** Fire when the value goes *above* a high threshold, but only recover when it drops *below* a separate, **lower** threshold. The gap between them means a value hovering at the firing line can't rapidly toggle on and off. (Same idea as a thermostat: heat on at 68°, off at 72°, not both at 70°.)

Together these give a three-state machine: **OK → PENDING → FIRING → OK**. We build it as a class you feed one measurement at a time; it returns an event only when the state actually changes.

In [ ]:
# --- Flap-resistant alert state machine --------------------------------------
class AlertState(Enum):
    OK = 'ok'
    PENDING = 'pending'
    FIRING = 'firing'

@dataclass
class AlertEvent:
    minute: int
    old_state: AlertState
    new_state: AlertState
    value: float

@dataclass
class AlertRule:
    name: str
    fire_threshold: float        # go PENDING/FIRING at or above this
    recover_threshold: float     # only leave FIRING below this (hysteresis)
    for_duration: int = 3        # steps the condition must hold before FIRING
    recover_duration: int = 3    # steps below recover_threshold before OK
    severity: str = 'ticket'
    state: AlertState = field(default=AlertState.OK, init=False)
    _above: int = field(default=0, init=False)
    _below: int = field(default=0, init=False)

    def update(self, minute, value):
        old = self.state
        if self.state in (AlertState.OK, AlertState.PENDING):
            if value >= self.fire_threshold:
                self._above += 1; self._below = 0
                if self.state == AlertState.OK:
                    self.state = AlertState.PENDING
                if self._above >= self.for_duration:
                    self.state = AlertState.FIRING
            else:
                self._above = 0
                self.state = AlertState.OK
        elif self.state == AlertState.FIRING:
            if value < self.recover_threshold:
                self._below += 1
                if self._below >= self.recover_duration:
                    self.state = AlertState.OK
                    self._above = 0; self._below = 0
            else:
                self._below = 0
        return AlertEvent(minute, old, self.state, value) if self.state != old else None

# Quick demo: a single 1-step spike should NOT fire (for_duration=3 absorbs it).
demo = AlertRule('demo', fire_threshold=0.10, recover_threshold=0.03,
                 for_duration=3, recover_duration=3)
spike = [0.0, 0.0, 0.20, 0.0, 0.0, 0.0]   # one lone spike
states = []
for i, v in enumerate(spike):
    demo.update(i, v); states.append(demo.state.value)
print('single-spike states:', states)
assert 'firing' not in states, 'a lone spike must be absorbed by for_duration'
print('Self-test passed: a one-sample spike never reaches FIRING.')

## 6. Severity routing: protecting the human

Not every firing condition deserves to **wake someone up**. The entire purpose of a severity level is to ration the scarcest resource you have — your on-call engineer's attention and trust.

| Severity | Action | When |
|---|---|---|
| **page** | Wake a human *now* | Fast, high burn rate confirmed on two windows — the SLO is about to blow |
| **ticket** | Handle in business hours | Slower burn, real but not urgent |
| **log** | Record only, no notification | Minor, self-healing, or informational |

The professional pattern is **multi-window, multi-burn-rate** (from the Google SRE workbook): only **page** when a *short* window shows a severe burn **and** a *longer* window confirms it isn't a one-off blip. This is what keeps a 30-second hiccup from paging someone while still catching a real, sustained outage fast.

In [ ]:
# --- Severity routing --------------------------------------------------------
def route(severity):
    table = {
        'page':   'PAGE on-call (wake a human now)',
        'ticket': 'FILE ticket (handle in business hours)',
        'log':    'LOG only (record, do not notify)',
    }
    return table.get(severity, 'LOG only (record, do not notify)')

def severity_for_burn(fast_burn, slow_burn):
    # page only when BOTH a short and a longer window agree it's severe
    if fast_burn >= 14.4 and slow_burn >= 6.0:
        return 'page'
    if fast_burn >= 3.0 and slow_burn >= 1.0:
        return 'ticket'
    return 'log'

# Show routing at a healthy minute and mid-incident using 5-min & 30-min windows.
for label, m in [('healthy (min 60)', 60), ('mid-incident (min 125)', 125)]:
    er_fast, _ = error_rate_in_window(EVENTS, m, 5)
    er_slow, _ = error_rate_in_window(EVENTS, m, 30)
    fb, sb = burn_rate(SLO_MAIN, er_fast), burn_rate(SLO_MAIN, er_slow)
    sev = severity_for_burn(fb, sb)
    print(f'{label:24s} fast={fb:6.1f}x slow={sb:6.1f}x -> {sev:6s} : {route(sev)}')

# Self-test: mid-incident should page; healthy should not.
er_f, _ = error_rate_in_window(EVENTS, 125, 5)
er_s, _ = error_rate_in_window(EVENTS, 125, 30)
assert severity_for_burn(burn_rate(SLO_MAIN, er_f), burn_rate(SLO_MAIN, er_s)) == 'page'
er_f, _ = error_rate_in_window(EVENTS, 60, 5)
er_s, _ = error_rate_in_window(EVENTS, 60, 30)
assert severity_for_burn(burn_rate(SLO_MAIN, er_f), burn_rate(SLO_MAIN, er_s)) == 'log'
print('\nSelf-test passed: incident pages, healthy traffic only logs.')

## 7. The payoff — run the full engine over the 3 hours

Now we tie it together. We walk minute by minute through the traffic, feed the rolling error rate into the `AlertRule` state machine, and record every state transition. Then we assert the three properties that separate a *good* alerting system from a noisy one:

1. **Sensitivity** — it FIRES during the incident.
2. **Specificity** — it does **not** fire during the healthy stretches (no false pages).
3. **Recovery** — it returns to OK shortly after the incident clears, on its own.

In [ ]:
# --- Run the alerting engine across the whole window -------------------------
rule = AlertRule(name='agent_error_burn',
                 fire_threshold=0.10,      # 10% rolling error rate
                 recover_threshold=0.03,   # hysteresis: recover under 3%
                 for_duration=3, recover_duration=3, severity='page')

transitions = []          # (minute, old, new, value)
firing_minutes = []       # minutes the rule was in FIRING
WIN = 5                   # rolling window width (minutes)

for m in range(WIN, DURATION_MIN):
    er, _ = error_rate_in_window(EVENTS, m, WIN)
    ev = rule.update(m, er)
    if ev:
        transitions.append(ev)
    if rule.state == AlertState.FIRING:
        firing_minutes.append(m)

# Render the transition timeline
tl = Table(title='Alert state transitions', show_lines=False)
tl.add_column('minute', justify='right'); tl.add_column('transition')
tl.add_column('rolling err', justify='right')
for ev in transitions:
    tl.add_row(str(ev.minute),
               f'{ev.old_state.value} -> {ev.new_state.value}',
               f'{ev.value:.1%}')
console.print(tl)

first_fire = min(firing_minutes) if firing_minutes else None
last_fire  = max(firing_minutes) if firing_minutes else None
print(f'\nFIRING from minute {first_fire} to {last_fire} '
      f'(incident was {INCIDENT_START}-{INCIDENT_END}).')

# ---- The three assertions that define a trustworthy alert ----
# 1. Sensitivity: it fired, and the first fire lands inside/just-after incident start
assert firing_minutes, 'engine must fire during the incident'
assert INCIDENT_START <= first_fire <= INCIDENT_START + WIN + rule.for_duration + 2, \
    'first fire should land right after the incident begins'
# 2. Specificity: no firing at all before the incident (no false pages)
assert all(m >= INCIDENT_START for m in firing_minutes), \
    'must NOT fire during healthy traffic'
# 3. Recovery: it clears within a reasonable tail after the incident ends
assert last_fire <= INCIDENT_END + WIN + rule.recover_duration + 3, \
    'alert should self-recover shortly after the incident clears'
print('All three properties hold: sensitive, specific, self-recovering.')

## 8. Package it: write a reusable `observability/alerting.py`

As in every Phase 8 lesson, we consolidate the concepts into an importable module and prove it works by importing it back. This `alerting.py` sits alongside the `tracing.py` / `online_eval.py` (L71) and `logging_search.py` (L72) modules — together they're the `observability/` package your `agent-bench` capstone (L76) will ship.

In [ ]:
import os, importlib, sys
os.makedirs(f'{BASE}/observability', exist_ok=True)
MODULE_CODE = r"""
'''
observability/alerting.py

Reusable SLO + alerting engine for LLM/agent systems.
Builds on the metrics (Lesson 71) and structured logs / trace search (Lesson 72)
you already emit. Nothing here needs an API key; it operates on events you log.

Core objects:
  SLO            - a reliability promise (target success rate over a window)
  error_budget   - how much failure the SLO allows, and how much is left
  burn_rate      - how fast you are spending that budget right now
  AlertRule      - a flap-resistant state machine (OK -> PENDING -> FIRING -> OK)
                   with a for-duration and hysteresis (separate recover threshold)
  Severity/route - decide whether a condition PAGES a human, files a TICKET, or LOGS
'''
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable, Optional


# ---- SLI / SLO / error budget -------------------------------------------------

@dataclass
class SLO:
    # A Service Level Objective: the promise. Example: 99% of requests succeed
    # over a rolling 30-day window. We keep the window in minutes for the demo.
    name: str
    target: float          # e.g. 0.99  (the SLI we promise to stay at or above)
    window_minutes: int    # the compliance window the target is measured over

    @property
    def allowed_error_rate(self) -> float:
        # The error budget expressed as a rate. SLO 99% -> 1% of requests may fail.
        return 1.0 - self.target


def sli_success_rate(n_total: int, n_bad: int) -> float:
    # A Service Level Indicator: a measured number. Here, fraction of good events.
    if n_total <= 0:
        return 1.0
    return 1.0 - (n_bad / n_total)


def error_budget(slo: SLO, n_total: int, n_bad: int) -> dict:
    # Budget = allowed failures over the window. Spent = actual failures.
    # Returned 'remaining_frac' < 0 means the SLO is already blown for this window.
    budget = slo.allowed_error_rate * n_total
    spent = float(n_bad)
    remaining = budget - spent
    remaining_frac = (remaining / budget) if budget > 0 else 0.0
    return {
        'budget_events': budget,
        'spent_events': spent,
        'remaining_events': remaining,
        'remaining_frac': remaining_frac,
        'sli': sli_success_rate(n_total, n_bad),
        'slo_met': sli_success_rate(n_total, n_bad) >= slo.target,
    }


def burn_rate(slo: SLO, window_error_rate: float) -> float:
    # Burn rate = how many times faster than 'sustainable' you are spending budget.
    # 1.0 = spending exactly at the pace that would exhaust the budget over the
    # full window. 14.4 over 1h = a whole 30-day budget gone in ~2 days.
    allowed = slo.allowed_error_rate
    if allowed <= 0:
        return float('inf') if window_error_rate > 0 else 0.0
    return window_error_rate / allowed


# ---- Alert state machine ------------------------------------------------------

class AlertState(Enum):
    OK = 'ok'            # condition not met
    PENDING = 'pending'  # condition met, but not yet for long enough to page
    FIRING = 'firing'    # condition sustained past for_duration -> alert is live


@dataclass
class AlertEvent:
    minute: int
    old_state: AlertState
    new_state: AlertState
    value: float


@dataclass
class AlertRule:
    # A flap-resistant rule. Two design ideas that separate a real alert from a
    # naive threshold check:
    #   for_duration     - the condition must hold this many steps before FIRING
    #                       (kills single-sample spikes)
    #   recover_threshold - a SEPARATE, lower threshold to leave FIRING (hysteresis)
    #                       so a value hovering at the line does not flap on/off
    name: str
    fire_threshold: float
    recover_threshold: float
    for_duration: int = 3
    recover_duration: int = 3
    severity: str = 'ticket'

    state: AlertState = field(default=AlertState.OK, init=False)
    _above_count: int = field(default=0, init=False)
    _below_count: int = field(default=0, init=False)

    def update(self, minute: int, value: float) -> Optional[AlertEvent]:
        # Feed one measurement. Returns an AlertEvent iff the state changed.
        old = self.state
        if self.state in (AlertState.OK, AlertState.PENDING):
            if value >= self.fire_threshold:
                self._above_count += 1
                self._below_count = 0
                if self.state == AlertState.OK:
                    self.state = AlertState.PENDING
                if self._above_count >= self.for_duration:
                    self.state = AlertState.FIRING
            else:
                self._above_count = 0
                self.state = AlertState.OK
        elif self.state == AlertState.FIRING:
            # Hysteresis: only leave FIRING once we drop under the LOWER
            # recover_threshold for recover_duration steps.
            if value < self.recover_threshold:
                self._below_count += 1
                if self._below_count >= self.recover_duration:
                    self.state = AlertState.OK
                    self._above_count = 0
                    self._below_count = 0
            else:
                self._below_count = 0

        if self.state != old:
            return AlertEvent(minute, old, self.state, value)
        return None


# ---- Severity routing ---------------------------------------------------------

def route(severity: str) -> str:
    # Where does a firing alert go? The whole point of severity is to protect
    # human attention: only real, urgent, actionable problems PAGE.
    table = {
        'page': 'PAGE on-call (wake a human now)',
        'ticket': 'FILE ticket (handle in business hours)',
        'log': 'LOG only (record, do not notify)',
    }
    return table.get(severity, 'LOG only (record, do not notify)')


def severity_for_burn(fast_burn: float, slow_burn: float) -> str:
    # Multi-window, multi-burn-rate (Google SRE). A fast, high burn on a short
    # window PAGES; a slower burn confirmed on a long window is a TICKET; small
    # burns just LOG. Requiring the long window too suppresses one-off blips.
    if fast_burn >= 14.4 and slow_burn >= 6.0:
        return 'page'
    if fast_burn >= 3.0 and slow_burn >= 1.0:
        return 'ticket'
    return 'log'

"""
with open(f'{BASE}/observability/alerting.py', 'w') as fh:
    fh.write(MODULE_CODE)
print('wrote', f'{BASE}/observability/alerting.py',
      '(%d bytes)' % len(MODULE_CODE))

# import it back and smoke-test
if BASE not in sys.path:
    sys.path.insert(0, BASE)
import observability.alerting as al
importlib.reload(al)
slo2 = al.SLO('reimport_check', 0.99, 180)
assert abs(slo2.allowed_error_rate - 0.01) < 1e-9
assert abs(al.burn_rate(slo2, 0.14) - 14.0) < 1e-9   # 14% err / 1% allowed
assert al.severity_for_burn(20.0, 8.0) == 'page'
assert al.severity_for_burn(0.5, 0.2) == 'log'
r = al.AlertRule('m', 0.10, 0.03, for_duration=3, recover_duration=3)
for v in [0.0, 0.0, 0.5]:            # only 1 sample above -> still PENDING
    r.update(0, v)
assert r.state.value == 'pending'
print('re-imported module smoke-test passed:',
      'SLO, burn_rate, severity routing, AlertRule all OK.')


## 9. Pitfalls — the ways alerting goes wrong in production

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | **Alerting on causes, not symptoms** | "CPU > 80%" pages you for something users never noticed. Alert on the SLI (user-facing success/latency), not the machine. |
| 2 | **Raw threshold, no `for` duration** | A 1-sample blip fires a page. Always require the condition to persist. |
| 3 | **No hysteresis** | A value at the threshold flaps on/off, producing an alert storm. Use a lower recovery threshold. |
| 4 | **Paging on everything** | Every non-actionable page erodes trust until on-call ignores the real one (alert fatigue). If it can wait, it's a ticket. |
| 5 | **Threshold with no SLO behind it** | "5% is bad" — says who? Burn rate ties the number to the promise so it's meaningful and portable. |
| 6 | **Single-window burn rate** | A short window alone is jumpy; a long window alone is slow. Require *both* (multi-window) to page. |
| 7 | **Symptom without a runbook** | A page with no "what do I do?" wastes the golden hour. Every paging alert needs a linked runbook. |
| 8 | **Alerting on absence of data as OK** | If the pipeline dies, error rate reads 0% and looks *healthy*. Alert on "no data" too. |
| 9 | **Static thresholds ignoring seasonality** | Traffic at 3am ≠ noon. A rate fine at peak is alarming at trough; consider relative/seasonal baselines. |
| 10 | **Never tuning alerts** | Every page should be reviewed: was it actionable? Delete or de-tune the ones that weren't. An alert nobody acts on is a bug. |

In [ ]:
# --- Verification checklist --------------------------------------------------
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))

# SLO / budget
check('SLI computed correctly', abs(sli_success_rate(100, 1) - 0.99) < 1e-9)
check('99% SLO -> 1% allowed error', abs(SLO_MAIN.allowed_error_rate - 0.01) < 1e-9)
check('incident blows the SLO', eb['slo_met'] is False)
check('error budget overspent', eb['remaining_events'] < 0)
# burn rate
check('burn_rate scales with error', abs(burn_rate(SLO_MAIN, 0.14) - 14.0) < 1e-9)
check('incident burn >> healthy burn',
      burn_rate(SLO_MAIN, er_inc) > 10 * max(burn_rate(SLO_MAIN, er_heal), 0.1))
# state machine
check('lone spike never fires', 'firing' not in states)
check('engine fired during incident', len(firing_minutes) > 0)
check('no false page before incident', all(m >= INCIDENT_START for m in firing_minutes))
check('alert self-recovers after incident',
      last_fire <= INCIDENT_END + WIN + rule.recover_duration + 3)
# routing
check('severe burn pages', severity_for_burn(20.0, 8.0) == 'page')
check('tiny burn only logs', severity_for_burn(0.5, 0.2) == 'log')
# module
import os as _os
check('alerting.py written to disk', _os.path.exists(f'{BASE}/observability/alerting.py'))

tbl = Table(title='Lesson 73 verification', show_lines=False)
tbl.add_column('check'); tbl.add_column('result', justify='right')
passed = 0
for name, ok in checks:
    tbl.add_row(name, '[green]PASS[/]' if ok else '[red]FAIL[/]')
    passed += int(ok)
console.print(tbl)
print(f'\n{passed}/{len(checks)} checks passed.')
assert passed == len(checks), 'some checks failed'
print('ALL CHECKS PASSED.')

## 10. Summary, homework & what's next

**What you built today**

| Concept | One-liner |
|---|---|
| **SLI / SLO / SLA** | measured indicator / your target / the customer contract |
| **Error budget** | `100% − SLO`; reliability is a budget you *spend*, not zero errors |
| **Burn rate** | `error_rate / allowed_error_rate`; ties any alert threshold to the promise |
| **`for` duration** | require the condition to persist → absorbs one-sample spikes |
| **Hysteresis** | fire high, recover low → no flapping at the threshold |
| **Multi-window burn** | page only when a short *and* a long window agree → fast but not jumpy |
| **Severity routing** | page vs ticket vs log → ration the human's attention |

**The one idea to remember:** *An alert that fires when nothing needs doing is a bug — it spends the only budget that matters more than your error budget: your on-call engineer's trust.*

**Homework (pick 1–2):**

1. **Add a latency SLO.** Define a second SLI — "p95 latency < 1500ms" — compute its own burn and wire a second `AlertRule`. Confirm the incident's latency inflation trips it too.
2. **Multi-window rule object.** Fold the fast+slow window logic *into* `AlertRule` so a single object owns both windows and emits the severity directly, instead of computing it separately.
3. **"No data" alert.** Add a rule that fires when a minute has *zero* events (pitfall #8) — prove it catches a simulated pipeline outage where you drop all events for 10 minutes.
4. **Budget-remaining alert.** Instead of instantaneous rate, alert when *projected* budget will hit zero before the window ends (a slow-burn "you'll miss the SLO by Friday" ticket).
5. **Wire it into `agent-bench`.** Feed the JSONL traces your L72 `TraceStore` already produces into this engine, so a benchmark run that regresses past your SLO auto-files a ticket.

**Next lesson — L74: A/B Testing & Guarded Rollouts.** You can now *detect* when production breaks. Next: how to safely *change* it — shipping a new prompt/model to 5% of traffic, comparing the two arms with a real statistical test (not vibes), and auto-rolling-back the moment the new arm burns budget faster than the old one. Alerting (today) becomes the guardrail that guards the rollout.